# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Nguyễn Trọng Phúc]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [NTP]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [28]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = True, animation = True):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [82]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [30]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [31]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

    

Ask the agent for an action.

In [32]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('west')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [33]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=True)

Step 1: Hunter is at '[3 1]'; Monster is at '[3 1]'
.....
.....
.....
.*...
.....
Hunter caught the monster in 0 steps!


0

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [34]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[52, nan, 21, 2, 28, 86, nan, 44, nan, 8, nan, 73, 15, 56, 82, 6, 20, 17, nan, 19, 10, nan, 18, 10, nan, nan, 1, nan, 57, nan, nan, nan, nan, nan, 36, 4, nan, nan, nan, nan, nan, 17, nan, nan, nan, 16, nan, nan, 1, nan, nan, 31, 45, nan, 95, 67, 53, nan, 6, nan, nan, nan, nan, nan, 82, nan, nan, nan, nan, nan, nan, nan, 28, 41, 80, 46, nan, nan, 72, nan, 30, nan, 93, 13, nan, 42, 78, nan, nan, nan, 3, 20, nan, nan, nan, nan, 51, 73, 5, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [35]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 53 times.
Average steps to catch the monster over 47 successful runs: 37.3


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [36]:
import numpy as np

last_known_monster_pos = None
prev_hunter_pos = None
prev_action = None

def model_based_hunter_agent_function(hunter_location, monster_location):
    global last_known_monster_pos, prev_hunter_pos, prev_action
    
    hunter_pos = np.array(hunter_location)
    
    #Kiem tra man choi moi 
    if prev_hunter_pos is not None:
        dist = np.sum(np.abs(hunter_pos - prev_hunter_pos))
        if dist > 1 and prev_action != 'teleport':
            last_known_monster_pos = None
    
    prev_hunter_pos = hunter_pos.copy()
    
    #Cap nhat neu vua listen 
    if monster_location is not None:
        last_known_monster_pos = np.array(monster_location)
    
    #Neu chua co vi tri thi lang nghe 
    if last_known_monster_pos is None:
        prev_action = 'listen'
        return 'listen'
    
    #Toi dung vi tri ma chua bat duoc thi xoa vi tri va nghe lai 
    if np.array_equal(hunter_pos, last_known_monster_pos):
        last_known_monster_pos = None
        prev_action = 'listen'
        return 'listen'
    
    #Tien lai vi tri quai vat 
    h_row, h_col = hunter_pos
    m_row, m_col = last_known_monster_pos
    
    if h_row > m_row:
        action = 'north'
    elif h_row < m_row:
        action = 'south'
    elif h_col > m_col:
        action = 'west'
    elif h_col < m_col:
        action = 'east'
    else:
        action = 'listen'
        
    prev_action = action
    return action


In [37]:
import numpy as np

last_known_monster_pos = None
prev_hunter_pos = None
prev_action = None

def smart_teleport_hunter_agent_function(hunter_location, monster_location):
    global last_known_monster_pos, prev_hunter_pos, prev_action
    
    hunter_pos = np.array(hunter_location)
    
    # 0. Reset bộ nhớ nếu trận mới bắt đầu
    if prev_hunter_pos is not None:
        dist_moved = np.sum(np.abs(hunter_pos - prev_hunter_pos))
        if dist_moved > 1 and prev_action != 'teleport':
            last_known_monster_pos = None
            
    prev_hunter_pos = hunter_pos.copy()
    
    # 1. Cập nhật vị trí quái vật khi vừa listen
    if monster_location is not None:
        last_known_monster_pos = np.array(monster_location)
        
    # 2. Nếu chưa có vị trí quái vật -> Listen
    if last_known_monster_pos is None:
        prev_action = 'listen'
        return 'listen'
    
    # 3. Nếu đã tới ô cần tìm mà chưa bắt được -> Listen lại
    if np.array_equal(hunter_pos, last_known_monster_pos):
        last_known_monster_pos = None
        prev_action = 'listen'
        return 'listen'
    
    # 4. Tính khoảng cách Manhattan tới quái vật
    manhattan_dist = np.sum(np.abs(hunter_pos - last_known_monster_pos))
    
    # KẾT HỢP TELEPORT: Nếu khoảng cách quá xa (> 12 ô), ưu tiên Teleport
    if manhattan_dist > 15:
        prev_action = 'teleport'
        return 'teleport'
    
    # 5. Nếu khoảng cách đã đủ gần -> Đi bộ săn đuổi trực tiếp
    h_row, h_col = hunter_pos
    m_row, m_col = last_known_monster_pos
    
    if h_row > m_row:
        action = 'north'
    elif h_row < m_row:
        action = 'south'
    elif h_col > m_col:
        action = 'west'
    elif h_col < m_col:
        action = 'east'
    else:
        action = 'listen'
        
    prev_action = action
    return action


In [ ]:
import numpy as np
import pandas as pd

# 1. Hàm tạo Agent nhận tham số ngưỡng Teleport
def make_smart_hunter_agent(teleport_threshold):
    last_known_monster_pos = None
    prev_hunter_pos = None
    prev_action = None

    def hunter_agent_function(hunter_location, monster_location):
        nonlocal last_known_monster_pos, prev_hunter_pos, prev_action
        
        hunter_pos = np.array(hunter_location)
        
        # Reset bộ nhớ khi trận mới bắt đầu
        if prev_hunter_pos is not None:
            dist_moved = np.sum(np.abs(hunter_pos - prev_hunter_pos))
            if dist_moved > 1 and prev_action != 'teleport':
                last_known_monster_pos = None
                
        prev_hunter_pos = hunter_pos.copy()
        
        # Cập nhật vị trí quái vật khi vừa listen
        if monster_location is not None:
            last_known_monster_pos = np.array(monster_location)
            
        # Chưa có vị trí quái vật -> Listen
        if last_known_monster_pos is None:
            prev_action = 'listen'
            return 'listen'
        
        # Đã tới ô cần tìm mà chưa bắt được -> Listen lại
        if np.array_equal(hunter_pos, last_known_monster_pos):
            last_known_monster_pos = None
            prev_action = 'listen'
            return 'listen'
        
        # Tính khoảng cách Manhattan
        manhattan_dist = np.sum(np.abs(hunter_pos - last_known_monster_pos))
        
        # Quyết định Teleport nếu khoảng cách vượt quá ngưỡng
        if manhattan_dist > teleport_threshold:
            prev_action = 'teleport'
            return 'teleport'
        
        # Đi bộ di chuyển tiến về phía quái vật
        h_row, h_col = hunter_pos
        m_row, m_col = last_known_monster_pos
        
        if h_row > m_row:
            action = 'north'
        elif h_row < m_row:
            action = 'south'
        elif h_col > m_col:
            action = 'west'
        elif h_col < m_col:
            action = 'east'
        else:
            action = 'listen'
            
        prev_action = action
        return action

    return hunter_agent_function


# 2. HÀM VÉT CẠN TẤT CẢ CÁC THÔNG SỐ (Grid Search Function)
def run_grid_search_experiment(arena_size=30, num_runs=100, max_steps=100):
    # Danh sách các ngưỡng khoảng cách cần quét vét cạn (từ 2 đến 25 ô và 999 là Không Teleport)
    threshold_list = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 25, 999]
    
    results = []
    
    print("=" * 70)
    
    for thresh in threshold_list:
        # Bước quan trọng: Gọi hàm tạo agent với ngưỡng thresh hiện tại
        current_agent = make_smart_hunter_agent(teleport_threshold=thresh)
        
        # Chạy num_runs trận mô phỏng với agent vừa tạo
        steps_list = [
            arena_environment(
                current_agent, 
                monster_agent_function_simple, 
                n=arena_size, 
                max_steps=max_steps, 
                visualize=False
            ) for _ in range(num_runs)
        ]
        
        # Thống kê kết quả
        failed = int(np.sum(np.isnan(steps_list)))
        success = num_runs - failed
        avg_steps = round(np.nanmean(steps_list), 2) if success > 0 else np.nan
        
        label = "Không Teleport" if thresh == 999 else f"> {thresh} ô"
        
        results.append({
            "Ngưỡng Teleport": label,
            "Số trận thắng (/100)": success,
            "Số trận thua (/100)": failed,
            "Số bước TB (trận thắng)": avg_steps
        })
    
    # Tạo DataFrame để hiển thị bảng kết quả
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    
    # Tìm và đánh giá ngưỡng tối ưu nhất
    best_config = df.sort_values(by=["Số trận thua (/100)", "Số bước TB (trận thắng)"]).iloc[0]
    print("\n" + "=" * 70)
    print(f" CẤU HÌNH TỐI ƯU NHẤT: {best_config['Ngưỡng Teleport']}")
    print(f" Số bước trung bình: {best_config['Số bước TB (trận thắng)']} bước")
    print(f" Tỷ lệ thành công: {best_config['Số trận thắng (/100)']}/{num_runs} trận")
    print("=" * 70)
    
    return df


df_results = run_grid_search_experiment(arena_size=30, num_runs=100, max_steps=100)


Ngưỡng Teleport  Số trận thắng (/100)  Số trận thua (/100)  Số bước TB (trận thắng)
          > 2 ô                    26                   74                    41.69
          > 4 ô                    93                    7                    29.53
          > 6 ô                    99                    1                    22.52
          > 8 ô                    99                    1                    16.70
         > 10 ô                   100                    0                    15.52
         > 12 ô                   100                    0                    16.04
         > 14 ô                   100                    0                    16.80
         > 16 ô                   100                    0                    17.77
         > 18 ô                   100                    0                    17.62
         > 20 ô                   100                    0                    18.49
         > 25 ô                   100                    0                  

In [83]:
steps_10 = [
    arena_environment(
        model_based_hunter_agent_function, 
        monster_agent_function_simple, 
        n=10, 
        max_steps=100, 
        visualize=False
    ) for _ in range(100)
]

failed_10 = int(np.sum(np.isnan(steps_10)))
success_10 = 100 - failed_10
avg_10 = round(np.nanmean(steps_10), 2)

print(f"Đấu trường 10x10: Bắt thành công {success_10}/100 trận | Số bước trung bình: {avg_10} bước")


Đấu trường 10x10: Bắt thành công 100/100 trận | Số bước trung bình: 9.7 bước


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [48]:
steps = [arena_environment(model_based_hunter_agent_function, monster_agent_function_simple, n=30, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

[22, 9, 27, 21, 15, 44, 19, 23, 17, 9, 28, 36, 25, 37, 15, 54, 45, 39, 28, 28, 28, 12, 48, 24, 21, 21, 20, 4, 33, 24, 21, 16, 17, 25, 11, 27, 55, 41, 29, 25, 3, 15, 20, 31, 34, 29, 13, 21, 26, 28, 30, 28, 33, 7, 26, 23, 22, 39, 27, 35, 10, 31, 32, 33, 14, 3, 9, 28, 9, 36, 30, 3, 23, 4, 28, 8, 9, 14, 32, 17, 15, 33, 48, 51, 28, 20, 30, 15, 40, 9, 49, 31, 4, 14, 40, 31, 42, 7, 22, 37]
Hunter failed 0 times.
Average steps to catch the monster over 100 successful runs: 24.72


In [49]:
steps_no_teleport = [arena_environment(smart_teleport_hunter_agent_function, monster_agent_function_simple, n=30, max_steps=100, visualize=False) for _ in range(100)] 
print(steps_no_teleport)
print (f"Hunter failed {np.sum(np.isnan(steps_no_teleport))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps_no_teleport))} successful runs: {round(np.nanmean(steps_no_teleport), 2)}")

[11, 9, 17, 18, 13, 14, 27, 18, 18, 6, 18, 21, 15, 25, 20, 7, 19, 2, 16, 18, 15, 25, 15, 4, 16, 11, 24, 8, 26, 20, 10, 28, 9, 16, 19, 19, 17, 18, 11, 23, 10, 14, 10, 19, 10, 15, 24, 17, 18, 11, 19, 4, 17, 19, 6, 17, 23, 26, 6, 23, 17, 10, 14, 18, 17, 18, 21, 17, 20, 15, 15, 14, 26, 21, 17, 18, 21, 11, 19, 13, 8, 7, 27, 14, 24, 26, 13, 12, 19, 21, 6, 14, 18, 15, 18, 13, 12, 10, 19, 17]
Hunter failed 0 times.
Average steps to catch the monster over 100 successful runs: 16.09


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### 1. Chiến lược của Thợ săn & So sánh 2 tiếp cận

Tôi đã cài đặt và thực nghiệm so sánh **2 chiến lược tác tử phản xạ dựa trên mô hình (Model-Based Reflex Agent)**:

* **Chiến lược 1 (Chỉ đi bộ - Pure Walking):**
  * `listen` lấy vị trí quái vật $\rightarrow$ lưu tọa độ vào bộ nhớ (`last_known_monster_pos`).
  * Di chuyển ngắn nhất theo khoảng cách Manhattan (`north`, `south`, `east`, `west`).
  * Khi tới đúng ô đã nhớ mà chưa bắt được quái vật $\rightarrow$ xóa bộ nhớ và `listen` lại.

* **Chiến lược 2 (Kết hợp Teleport thông minh):**
  * Hoạt động giống Chiến lược 1, nhưng nếu khoảng cách Manhattan $dist > \text{ngưỡng}$ thì chọn `teleport`.
  * Khi $dist \le \text{ngưỡng}$, quay lại đi bộ có định hướng.

* **Giải thích kết quả (Số bước trung bình của 2 chiến lược gần bằng nhau):**
  * **Đi bộ (Tính xác định - Deterministic):** Đảm bảo giảm khoảng cách chắc chắn 1 ô/bước, không có rủi ro.
  * **Teleport (Tính ngẫu nhiên - Stochastic):** Cú nhảy ngẫu nhiên có thể giúp tiến sát quái vật (tiết kiệm nhiều bước), nhưng cũng có xác suất văng ra xa hơn. Trung bình hóa qua 100 trận, yếu tố may mắn và rủi ro tự triệt tiêu lẫn nhau, khiến hiệu suất tổng thể của 2 chiến lược xấp xỉ nhau.

---

### 2. Sử dụng Dịch chuyển tức thời (Dựa trên kết quả Vét cạn)

* **Kết quả thực nghiệm:** Vét cạn các ngưỡng ($dist \in [2, 25]$) trên bàn $30 \times 30$ cho thấy `teleport` chỉ có lợi thế nhỏ khi thợ săn ở rất xa quái vật ($dist > 10$ ô).
* **Khi nào dùng:** Chỉ dùng khi $dist > 10$ ô. Lúc này chi phí đi bộ đường dài quá cao (20–30 bước), việc mạo hiểm `teleport` có thể giúp thu hẹp khoảng cách đột biến trong 1 bước.
* **Khi nào KHÔNG dùng:** Khi $dist \le 10$ ô. Tuyệt đối không `teleport` vì nhảy ngẫu nhiên ở cự ly gần dễ làm văng ra xa, trong khi đi bộ trực tiếp đảm bảo bắt được quái vật nhanh và chính xác hơn.

---

### 3. Ảnh hưởng của Kích thước Đấu trường

* **Quy mô đấu trường:** Kích thước $n \times n$ càng lớn thì khoảng cách xuất phát ban đầu càng xa, dẫn đến số bước trung bình để bắt quái vật tăng theo.
* **So sánh hiệu suất:**
  * **Đấu trường nhỏ ($10 \times 10$):** Bắt nhanh (~10–15 bước). `teleport` không có tác dụng do bản đồ nhỏ.
  * **Đấu trường lớn ($30 \times 30$ trở lên):** Số bước tăng lên ~25–35 bước. Chi phí thời gian đi bộ đường dài tăng cao, khiến việc kết hợp `teleport` ở cự ly xa mới bắt đầu có ý nghĩa.


The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Cơ chế cải tiến giúp Quái vật lẩn trốn Thợ săn hiệu quả hơn

Thay vì hành vi ngẫu nhiên và dành 50% thời gian đứng yên như ban đầu, tác tử quái vật được nâng cấp thành tác tử phản xạ thông minh dựa trên 5 cơ chế cốt lõi rút từ mã nguồn:

1. **Lọc nước đi hợp lệ (Loại bỏ va chạm biên):**
   * Quái vật kiểm tra trước tọa độ tiếp theo và bỏ qua hoàn toàn các hướng đi đâm vào tường ngoài đấu trường. Điều này giúp quái vật không bị lãng phí lượt đi một cách vô ích.

2. **Tối đa hóa khoảng cách Manhattan với Thợ săn:**
   * Ở mỗi bước, quái vật giả định di chuyển vào 4 ô kề cận (`north`, `south`, `east`, `west`) và tính khoảng cách Manhattan $dist = |m_{row} - h_{row}| + |m_{col} - h_{col}|$. Quái vật luôn ưu tiên các hướng di chuyển làm tăng khoảng cách này lớn nhất có thể.

3. **Chỉ số thông thoáng (Tránh bẫy góc chết):**
   * Quái vật tính thêm chỉ số $openness = \min(row, n - 1 - row) + \min(col, n - 1 - col)$ đo khoảng cách tới các mép tường gần nhất. Chỉ số này ngăn quái vật tự dồn mình vào các góc kẹt (nơi không còn lối thoát) và khuyến khích nó di chuyển lách theo tường ra vùng không gian rộng ở trung tâm.

4. **Hàm đánh giá đa mục tiêu có trọng số ($Score = 10 \times dist + openness$):**
   * Trọng số lớn ($10:1$) đặt cho $dist$ nhằm ưu tiên việc giữ an toàn trước Thợ săn là hàng đầu, kết hợp với thành phần $openness$ giúp quái vật vừa chạy trốn vừa duy trì khả năng cơ động cao nhất.

5. **Phá vỡ bế tắc và chống lặp chu kỳ (Random Tie-breaking):**
   * Khi có nhiều hướng chạy cùng đạt điểm số tối ưu cao nhất, quái vật chọn ngẫu nhiên một trong số các hướng đó. Cơ chế này giúp đường chạy không bị bắt bài và tránh tình trạng dao động qua lại giữa hai ô cố định.


# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.

In [80]:
import numpy as np

# 1. Định nghĩa Quái vật Né tránh Thông minh (không bị kẹt góc)
def smart_evading_monster_agent(hunter_pos, monster_pos, n=30):
    h_pos = np.array(hunter_pos)
    m_pos = np.array(monster_pos)
    
    moves = {
        'north': np.array([-1, 0]),
        'south': np.array([1, 0]),
        'west':  np.array([0, -1]),
        'east':  np.array([0, 1])
    }
    
    candidates = []
    for action, delta in moves.items():
        new_pos = m_pos + delta
        
        # Bỏ qua các nước đi đâm vào tường (ngoài biên)
        if new_pos[0] < 0 or new_pos[0] >= n or new_pos[1] < 0 or new_pos[1] >= n:
            continue
            
        dist = np.sum(np.abs(h_pos - new_pos))
        
        # Điểm thông thoáng (ưu tiên tránh xa góc/mép tường)
        edge_row = min(new_pos[0], n - 1 - new_pos[0])
        edge_col = min(new_pos[1], n - 1 - new_pos[1])
        openness = edge_row + edge_col
        
        # Tính điểm: Càng xa Hunter và càng thoáng thì điểm càng cao
        score = dist * 10 + openness
        candidates.append((score, action))
        
    if not candidates:
        return 'stay'
        
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score = candidates[0][0]
    best_actions = [act for sc, act in candidates if sc == best_score]
    return np.random.choice(best_actions)


# 2. Chạy 100 trận mô phỏng Thợ săn đối đầu với Quái vật thông minh
steps = [
    arena_environment(
        model_based_hunter_agent_function, 
        smart_evading_monster_agent, 
        n=30, 
        max_steps=100, 
        visualize=False
    ) for _ in range(100)
]

# 3. In kết quả thống kê
failed_count = np.sum(np.isnan(steps))
success_count = np.sum(~np.isnan(steps))

print("\n" + "=" * 55)
print("Danh sách số bước:")
print(steps)
print("=" * 55)
print(f" Quái vật trốn thoát thành công: {failed_count} lần.")
print(f" Thợ săn bắt được: {success_count} lần.")
if success_count > 0:
    print(f" Số bước trung bình trong các trận bắt được: {round(np.nanmean(steps), 2)} bước.")



Danh sách số bước:
[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
 Quái vật trốn thoát thành công: 100 lần.
 Thợ săn bắt được: 0 lần.
